In [22]:
# Reload corrected PolygonInferenceService after editing app/services/polygon_inference_service.py

from importlib import reload

import app.services.polygon_inference_service as polygon_service_module

reload(polygon_service_module)

PolygonInferenceService = polygon_service_module.PolygonInferenceService

service = PolygonInferenceService(master_df=master_df)

print("Reloaded PolygonInferenceService with target-month subsystem selection fix.")

Reloaded PolygonInferenceService with target-month subsystem selection fix.


In [21]:
import sys
from pathlib import Path

import ee
import pandas as pd

# tests/backtests.ipynb → RozviDrought project root
PROJECT_ROOT = Path.cwd().resolve()

# If notebook is launched from tests/, move one level up.
if PROJECT_ROOT.name == "tests":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

EE_PROJECT = "august-analyze"

try:
    ee.Initialize(project=EE_PROJECT)
    print(f"Earth Engine initialized with project: {EE_PROJECT}")
except Exception as exc:
    print("Earth Engine not initialized. Running authentication...")
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
    print(f"Earth Engine authenticated and initialized with project: {EE_PROJECT}")

print("Project root:", PROJECT_ROOT)

Earth Engine initialized with project: august-analyze
Project root: C:\Projects\Infer RozviDrought\RozviDrought


In [2]:
# Cell 2: Load Zimbabwe administrative polygons from Earth Engine

ADMIN_LEVEL = 2
GAUL_ASSET = f"FAO/GAUL/2015/level{ADMIN_LEVEL}"

admin_fc = (
    ee.FeatureCollection(GAUL_ASSET)
    .filter(ee.Filter.eq("ADM0_NAME", "Zimbabwe"))
)

admin_count = admin_fc.size().getInfo()

first_admin = admin_fc.first().toDictionary().getInfo()

print(f"Loaded Zimbabwe ADM{ADMIN_LEVEL} polygons from:", GAUL_ASSET)
print("Admin polygon count:", admin_count)
print("Example properties:")
first_admin

Loaded Zimbabwe ADM2 polygons from: FAO/GAUL/2015/level2
Admin polygon count: 62
Example properties:


{'ADM0_CODE': 271,
 'ADM0_NAME': 'Zimbabwe',
 'ADM1_CODE': 3436,
 'ADM1_NAME': 'Harare',
 'ADM2_CODE': 68807,
 'ADM2_NAME': 'Chitungwiza',
 'DISP_AREA': 'NO',
 'EXP2_YEAR': 3000,
 'STATUS': 'Member State',
 'STR2_YEAR': 2006,
 'Shape_Area': 0.00407260399874,
 'Shape_Leng': 0.34197890796}

In [3]:
# Cell 3: Pull Zimbabwe ADM2 polygons from Earth Engine into local records

admin_features = admin_fc.getInfo()["features"]

admin_records = []

for feature in admin_features:
    props = feature["properties"]
    geom = feature["geometry"]

    admin_records.append({
        "adm0_name": props.get("ADM0_NAME"),
        "adm1_name": props.get("ADM1_NAME"),
        "adm2_name": props.get("ADM2_NAME"),
        "adm2_code": props.get("ADM2_CODE"),
        "geometry": geom,
    })

admin_df = pd.DataFrame(admin_records)

print("Admin records loaded:", len(admin_df))
print(admin_df[["adm1_name", "adm2_name", "adm2_code"]].head())

Admin records loaded: 62
             adm1_name    adm2_name  adm2_code
0               Harare  Chitungwiza      68807
1               Harare       Harare      68809
2  Mashonaland Central       Guruve      68808
3  Mashonaland Central        Mbire      68811
4           Manicaland       Makoni      33056


In [4]:
# Cell 4: Convert ADM2 GeoJSON geometries to Shapely geometries

from shapely.geometry import shape

admin_df["shapely_geometry"] = admin_df["geometry"].apply(shape)

print("Converted ADM2 geometries to Shapely.")
print(admin_df[["adm1_name", "adm2_name", "adm2_code"]].head())
print("Example geometry type:", admin_df.loc[0, "shapely_geometry"].geom_type)

Converted ADM2 geometries to Shapely.
             adm1_name    adm2_name  adm2_code
0               Harare  Chitungwiza      68807
1               Harare       Harare      68809
2  Mashonaland Central       Guruve      68808
3  Mashonaland Central        Mbire      68811
4           Manicaland       Makoni      33056
Example geometry type: Polygon


In [5]:
# Cell 5: Load master model inputs and initialize polygon inference service

from app.services.polygon_inference_service import PolygonInferenceService

WORKSPACE_DIR = PROJECT_ROOT.parent

MASTER_PATH = (
    WORKSPACE_DIR
    / "data"
    / "master_inputs"
    / "master_inputs_long_198001_205012.parquet"
)

print("Using dataset:", MASTER_PATH)

master_df = pd.read_parquet(MASTER_PATH)

service = PolygonInferenceService(master_df=master_df)

print("Master rows:", len(master_df))
print("Master columns:", len(master_df.columns))
print("PolygonInferenceService initialized.")

Using dataset: C:\Projects\Infer RozviDrought\data\master_inputs\master_inputs_long_198001_205012.parquet
Master rows: 38957625
Master columns: 13
PolygonInferenceService initialized.


In [6]:
# Cell 7: Hardcode drought timeline and expand to monthly backtest targets

DROUGHT_EVENTS = [
    {"event_id": 1, "period": "1902-1903", "season": "1902/03", "start_year": 1902, "end_year": 1903, "duration_months": 12, "timeline_severity": 2, "severity_label": "2 - Moderate"},
    {"event_id": 2, "period": "1911-1912", "season": "1911/12", "start_year": 1911, "end_year": 1912, "duration_months": 10, "timeline_severity": 2, "severity_label": "2 - Moderate"},
    {"event_id": 3, "period": "1921-1922", "season": "1921/22", "start_year": 1921, "end_year": 1922, "duration_months": 10, "timeline_severity": 2, "severity_label": "2 - Moderate"},
    {"event_id": 4, "period": "1932-1933", "season": "1932/33", "start_year": 1932, "end_year": 1933, "duration_months": 12, "timeline_severity": 3, "severity_label": "3 - Severe"},
    {"event_id": 5, "period": "1946-1947", "season": "1946/47", "start_year": 1946, "end_year": 1947, "duration_months": 11, "timeline_severity": 3, "severity_label": "3 - Severe"},
    {"event_id": 6, "period": "1967-1968", "season": "1967/68", "start_year": 1967, "end_year": 1968, "duration_months": 10, "timeline_severity": 2, "severity_label": "2 - Moderate"},
    {"event_id": 7, "period": "1972-1973", "season": "1972/73", "start_year": 1972, "end_year": 1973, "duration_months": 11, "timeline_severity": 3, "severity_label": "3 - Severe"},
    {"event_id": 8, "period": "1982-1984", "season": "1982-84", "start_year": 1982, "end_year": 1984, "duration_months": 24, "timeline_severity": 3, "severity_label": "3 - Severe"},
    {"event_id": 9, "period": "1986-1987", "season": "1986/87", "start_year": 1986, "end_year": 1987, "duration_months": 10, "timeline_severity": 2, "severity_label": "2 - Moderate"},
    {"event_id": 10, "period": "1991-1992", "season": "1991/92", "start_year": 1991, "end_year": 1992, "duration_months": 12, "timeline_severity": 5, "severity_label": "5 - Catastrophic"},
    {"event_id": 11, "period": "1994-1995", "season": "1994/95", "start_year": 1994, "end_year": 1995, "duration_months": 10, "timeline_severity": 3, "severity_label": "3 - Severe"},
    {"event_id": 12, "period": "1997-1998", "season": "1997/98", "start_year": 1997, "end_year": 1998, "duration_months": 12, "timeline_severity": 2, "severity_label": "2 - Moderate"},
    {"event_id": 13, "period": "2001-2002", "season": "2001/02 + 2002/03", "start_year": 2001, "end_year": 2002, "duration_months": 18, "timeline_severity": 4, "severity_label": "4 - Extreme"},
    {"event_id": 14, "period": "2012-2013", "season": "2012/13", "start_year": 2012, "end_year": 2013, "duration_months": 8, "timeline_severity": 2, "severity_label": "2 - Moderate"},
    {"event_id": 15, "period": "2015-2016", "season": "2015/16", "start_year": 2015, "end_year": 2016, "duration_months": 12, "timeline_severity": 4, "severity_label": "4 - Extreme"},
    {"event_id": 16, "period": "2018-2019", "season": "2018/19", "start_year": 2018, "end_year": 2019, "duration_months": 8, "timeline_severity": 2, "severity_label": "2 - Moderate"},
    {"event_id": 17, "period": "2023-2024", "season": "2023/24", "start_year": 2023, "end_year": 2024, "duration_months": 12, "timeline_severity": 5, "severity_label": "5 - Catastrophic"},
]

yyyymm_col = "yyyymm" if "yyyymm" in master_df.columns else "date"

available_months = set(master_df[yyyymm_col].astype(int).unique())

monthly_rows = []

for event in DROUGHT_EVENTS:
    for year in range(event["start_year"], event["end_year"] + 1):
        for month in range(1, 13):
            yyyymm = int(f"{year}{month:02d}")

            monthly_rows.append({
                **event,
                "year": year,
                "month": month,
                "yyyymm": yyyymm,
                "available_in_master": yyyymm in available_months,
            })

backtest_months_df = pd.DataFrame(monthly_rows)

available_backtest_months_df = backtest_months_df[
    backtest_months_df["available_in_master"]
].reset_index(drop=True)

print("Total expanded event-month rows:", len(backtest_months_df))
print("Available event-month rows in master_df:", len(available_backtest_months_df))
print("First available rows:")
print(available_backtest_months_df.head(12))

Total expanded event-month rows: 420
Available event-month rows in master_df: 252
First available rows:
    event_id     period   season  start_year  end_year  duration_months  \
0          8  1982-1984  1982-84        1982      1984               24   
1          8  1982-1984  1982-84        1982      1984               24   
2          8  1982-1984  1982-84        1982      1984               24   
3          8  1982-1984  1982-84        1982      1984               24   
4          8  1982-1984  1982-84        1982      1984               24   
5          8  1982-1984  1982-84        1982      1984               24   
6          8  1982-1984  1982-84        1982      1984               24   
7          8  1982-1984  1982-84        1982      1984               24   
8          8  1982-1984  1982-84        1982      1984               24   
9          8  1982-1984  1982-84        1982      1984               24   
10         8  1982-1984  1982-84        1982      1984               24

In [9]:
# Cell 8: Safe ADM2 drought backtest smoke test

import gc
import time

SCENARIO = "historical"
MODEL = "hybrid"

# Keep this tiny for the smoke test.
MAX_ADMINS = 1
MAX_MONTHS = 1

# Prefer small ADM2 polygons first to reduce pixel load.
admin_test_df = admin_df.copy()
admin_test_df["area_deg2"] = admin_test_df["shapely_geometry"].apply(lambda g: g.area)
admin_test_df = admin_test_df.sort_values("area_deg2").head(MAX_ADMINS).reset_index(drop=True)

# Use latest available drought months first because recent data is more likely complete.
month_test_df = (
    available_backtest_months_df
    .sort_values("yyyymm", ascending=False)
    .head(MAX_MONTHS)
    .reset_index(drop=True)
)

print("Safe smoke test scope")
print("Admins tested:", len(admin_test_df))
print("Months tested:", len(month_test_df))
print("Selected admin:")
print(admin_test_df[["adm1_name", "adm2_name", "adm2_code", "area_deg2"]])
print("Selected month:")
print(month_test_df[["event_id", "period", "yyyymm", "timeline_severity", "severity_label"]])

smoke_result = None
smoke_error = None

for admin_i, admin_row in admin_test_df.iterrows():
    for month_i, event_row in month_test_df.iterrows():
        started = time.time()

        print(
            f"\nRunning smoke inference "
            f"admin {admin_i + 1}/{len(admin_test_df)} | "
            f"month {month_i + 1}/{len(month_test_df)} | "
            f"{admin_row['adm2_name']} | {int(event_row['yyyymm'])}"
        )

        try:
            result = service.infer_polygon(
                geometry=admin_row["shapely_geometry"],
                scenario=SCENARIO,
                yyyymm=int(event_row["yyyymm"]),
                model=MODEL,
            )

            smoke_result = {
                "admin": admin_row[["adm1_name", "adm2_name", "adm2_code"]].to_dict(),
                "event": event_row[[
                    "event_id",
                    "period",
                    "season",
                    "yyyymm",
                    "timeline_severity",
                    "severity_label",
                ]].to_dict(),
                "summary": result.summary,
                "cell_result_count": len(result.cell_results),
            }

            print("Success.")
            print("Elapsed seconds:", round(time.time() - started, 2))
            break

        except Exception as exc:
            smoke_error = {
                "admin": admin_row[["adm1_name", "adm2_name", "adm2_code"]].to_dict(),
                "yyyymm": int(event_row["yyyymm"]),
                "error_type": type(exc).__name__,
                "error": str(exc),
            }

            print("Failed.")
            print("Elapsed seconds:", round(time.time() - started, 2))
            print("Error type:", smoke_error["error_type"])
            print("Error:", smoke_error["error"])

        finally:
            gc.collect()

    if smoke_result is not None:
        break

if smoke_result is None:
    print("\nNo successful smoke result.")
    print("Last error:")
    print(smoke_error)
else:
    print("\nSmoke result:")
    print(smoke_result)

Safe smoke test scope
Admins tested: 1
Months tested: 1
Selected admin:
  adm1_name    adm2_name  adm2_code  area_deg2
0    Harare  Chitungwiza      68807   0.004073
Selected month:
   event_id     period  yyyymm  timeline_severity    severity_label
0        17  2023-2024  202412                  5  5 - Catastrophic

Running smoke inference admin 1/1 | month 1/1 | Chitungwiza | 202412

Success.
Elapsed seconds: 17.67

Smoke result:
{'admin': {'adm1_name': 'Harare', 'adm2_name': 'Chitungwiza', 'adm2_code': 68807}, 'event': {'event_id': 17, 'period': '2023-2024', 'season': '2023/24', 'yyyymm': 202412, 'timeline_severity': 5, 'severity_label': '5 - Catastrophic'}, 'summary': {'cells_inferred': 3, 'mean_confidence': 0.9960067669550577, 'dominant_class': 'normal'}, 'cell_result_count': 3}


In [23]:
# Cell 9: Safe single-admin monthly validation for 2023/24 drought season

import gc
import time

SCENARIO = "historical"
MODEL = "hybrid"

TEST_ADM2_NAME = "Chitungwiza"
TEST_MONTHS = [202310, 202311, 202312, 202401, 202402, 202403, 202404]

admin_match = admin_df[admin_df["adm2_name"].eq(TEST_ADM2_NAME)]

if admin_match.empty:
    raise ValueError(f"Could not find ADM2 polygon: {TEST_ADM2_NAME}")

admin_row = admin_match.iloc[0]
test_geometry = admin_row["shapely_geometry"]

event_match = available_backtest_months_df[
    available_backtest_months_df["yyyymm"].isin(TEST_MONTHS)
].copy()

print("Monthly validation scope")
print("Admin:", admin_row[["adm1_name", "adm2_name", "adm2_code"]].to_dict())
print("Requested months:", TEST_MONTHS)
print("Available months:", event_match["yyyymm"].tolist())

monthly_summary_rows = []

for i, event_row in event_match.sort_values("yyyymm").iterrows():
    yyyymm = int(event_row["yyyymm"])
    started = time.time()

    print(f"\nRunning {TEST_ADM2_NAME} | {yyyymm}")

    try:
        result = service.infer_polygon(
            geometry=test_geometry,
            scenario=SCENARIO,
            yyyymm=yyyymm,
            model=MODEL,
        )

        summary = dict(result.summary)

        monthly_summary_rows.append({
            "adm1_name": admin_row["adm1_name"],
            "adm2_name": admin_row["adm2_name"],
            "adm2_code": admin_row["adm2_code"],
            "event_id": int(event_row["event_id"]),
            "period": event_row["period"],
            "season": event_row["season"],
            "yyyymm": yyyymm,
            "timeline_severity": int(event_row["timeline_severity"]),
            "severity_label": event_row["severity_label"],
            "status": "success",
            "cells_inferred": summary.get("cells_inferred"),
            "mean_confidence": summary.get("mean_confidence"),
            "dominant_class": summary.get("dominant_class"),
            "elapsed_seconds": round(time.time() - started, 2),
            "error": None,
        })

        print("Success:", summary)
        print("Elapsed seconds:", round(time.time() - started, 2))

    except Exception as exc:
        monthly_summary_rows.append({
            "adm1_name": admin_row["adm1_name"],
            "adm2_name": admin_row["adm2_name"],
            "adm2_code": admin_row["adm2_code"],
            "event_id": int(event_row["event_id"]),
            "period": event_row["period"],
            "season": event_row["season"],
            "yyyymm": yyyymm,
            "timeline_severity": int(event_row["timeline_severity"]),
            "severity_label": event_row["severity_label"],
            "status": "failed",
            "cells_inferred": None,
            "mean_confidence": None,
            "dominant_class": None,
            "elapsed_seconds": round(time.time() - started, 2),
            "error": f"{type(exc).__name__}: {exc}",
        })

        print("Failed:", type(exc).__name__, exc)
        print("Elapsed seconds:", round(time.time() - started, 2))

    finally:
        gc.collect()

monthly_validation_df = pd.DataFrame(monthly_summary_rows)

print("\nMonthly validation results")
print(monthly_validation_df)

Monthly validation scope
Admin: {'adm1_name': 'Harare', 'adm2_name': 'Chitungwiza', 'adm2_code': 68807}
Requested months: [202310, 202311, 202312, 202401, 202402, 202403, 202404]
Available months: [202310, 202311, 202312, 202401, 202402, 202403, 202404]

Running Chitungwiza | 202310
Success: {'cells_inferred': 3, 'mean_confidence': 0.48375733693440753, 'dominant_class': 'moderate'}
Elapsed seconds: 7.59

Running Chitungwiza | 202311
Success: {'cells_inferred': 3, 'mean_confidence': 0.7469316522280375, 'dominant_class': 'normal'}
Elapsed seconds: 7.52

Running Chitungwiza | 202312
Success: {'cells_inferred': 3, 'mean_confidence': 0.911540687084198, 'dominant_class': 'normal'}
Elapsed seconds: 6.97

Running Chitungwiza | 202401
Success: {'cells_inferred': 3, 'mean_confidence': 0.967751145362854, 'dominant_class': 'normal'}
Elapsed seconds: 7.13

Running Chitungwiza | 202402
Success: {'cells_inferred': 3, 'mean_confidence': 0.9971097707748413, 'dominant_class': 'normal'}
Elapsed seconds: 

In [11]:
# Cell 10: Inspect cell-level outputs across 2023/24 months

cell_debug_rows = []

for _, event_row in event_match.sort_values("yyyymm").iterrows():
    yyyymm = int(event_row["yyyymm"])

    print(f"Inspecting cell outputs for {TEST_ADM2_NAME} | {yyyymm}")

    result = service.infer_polygon(
        geometry=test_geometry,
        scenario=SCENARIO,
        yyyymm=yyyymm,
        model=MODEL,
    )

    for cell_i, cell_result in enumerate(result.cell_results):
        row = {
            "yyyymm": yyyymm,
            "cell_i": cell_i,
            "timeline_severity": int(event_row["timeline_severity"]),
            "severity_label": event_row["severity_label"],
        }

        if isinstance(cell_result, dict):
            for key, value in cell_result.items():
                if key != "fusion_input":
                    row[key] = value

        cell_debug_rows.append(row)

    gc.collect()

cell_debug_df = pd.DataFrame(cell_debug_rows)

print("Cell debug rows:", len(cell_debug_df))
print("Columns:")
print(list(cell_debug_df.columns))
print("\nPreview:")
print(cell_debug_df.head(20))

Inspecting cell outputs for Chitungwiza | 202310
Inspecting cell outputs for Chitungwiza | 202311
Inspecting cell outputs for Chitungwiza | 202312
Inspecting cell outputs for Chitungwiza | 202401
Inspecting cell outputs for Chitungwiza | 202402
Inspecting cell outputs for Chitungwiza | 202403
Inspecting cell outputs for Chitungwiza | 202404
Cell debug rows: 21
Columns:
['yyyymm', 'cell_i', 'timeline_severity', 'severity_label', 'pixel_id', 'lon', 'lat', 'result']

Preview:
    yyyymm  cell_i  timeline_severity    severity_label  pixel_id        lon  \
0   202310       0                  5  5 - Catastrophic      9405  31.059251   
1   202310       1                  5  5 - Catastrophic      9580  31.059251   
2   202310       2                  5  5 - Catastrophic      9581  31.104167   
3   202311       0                  5  5 - Catastrophic      9405  31.059251   
4   202311       1                  5  5 - Catastrophic      9580  31.059251   
5   202311       2                  5  5 -

In [12]:
# Cell 11: Flatten nested fusion outputs for monthly comparison

flattened_rows = []

for _, row in cell_debug_df.iterrows():
    result_obj = row["result"]

    flat = {
        "yyyymm": row["yyyymm"],
        "cell_i": row["cell_i"],
        "pixel_id": row["pixel_id"],
        "lon": row["lon"],
        "lat": row["lat"],
        "timeline_severity": row["timeline_severity"],
        "severity_label": row["severity_label"],
    }

    if isinstance(result_obj, dict):
        fusion_result = result_obj.get("fusion_result")
        fusion_input = result_obj.get("fusion_input")

        flat["fusion_input_type"] = type(fusion_input).__name__
        flat["fusion_result_type"] = type(fusion_result).__name__

        if isinstance(fusion_result, dict):
            for key, value in fusion_result.items():
                flat[f"fusion_{key}"] = value
        else:
            flat["fusion_result_raw"] = str(fusion_result)

    flattened_rows.append(flat)

flat_cell_debug_df = pd.DataFrame(flattened_rows)

print("Flattened rows:", len(flat_cell_debug_df))
print("Columns:")
print(list(flat_cell_debug_df.columns))
print("\nPreview:")
print(flat_cell_debug_df.head(21))

Flattened rows: 21
Columns:
['yyyymm', 'cell_i', 'pixel_id', 'lon', 'lat', 'timeline_severity', 'severity_label', 'fusion_input_type', 'fusion_result_type', 'fusion_result_raw']

Preview:
    yyyymm  cell_i  pixel_id        lon        lat  timeline_severity  \
0   202310       0      9405  31.059251 -17.988764                  5   
1   202310       1      9580  31.059251 -18.033679                  5   
2   202310       2      9581  31.104167 -18.033679                  5   
3   202311       0      9405  31.059251 -17.988764                  5   
4   202311       1      9580  31.059251 -18.033679                  5   
5   202311       2      9581  31.104167 -18.033679                  5   
6   202312       0      9405  31.059251 -17.988764                  5   
7   202312       1      9580  31.059251 -18.033679                  5   
8   202312       2      9581  31.104167 -18.033679                  5   
9   202401       0      9405  31.059251 -17.988764                  5   
10  20240

In [13]:
# Cell 12: Extract PredictionResult attributes from nested cell outputs

prediction_rows = []

for _, row in cell_debug_df.iterrows():
    result_obj = row["result"]
    fusion_result = result_obj.get("fusion_result") if isinstance(result_obj, dict) else None
    fusion_input = result_obj.get("fusion_input") if isinstance(result_obj, dict) else None

    out = {
        "yyyymm": row["yyyymm"],
        "cell_i": row["cell_i"],
        "pixel_id": row["pixel_id"],
        "lon": row["lon"],
        "lat": row["lat"],
        "timeline_severity": row["timeline_severity"],
        "severity_label": row["severity_label"],
    }

    if hasattr(fusion_result, "__dict__"):
        for key, value in fusion_result.__dict__.items():
            out[key] = value
    else:
        out["fusion_result_raw"] = str(fusion_result)

    if isinstance(fusion_input, pd.DataFrame) and not fusion_input.empty:
        for col, value in fusion_input.iloc[0].to_dict().items():
            out[f"input_{col}"] = value

    prediction_rows.append(out)

prediction_debug_df = pd.DataFrame(prediction_rows)

print("Prediction debug rows:", len(prediction_debug_df))
print("Columns:")
print(list(prediction_debug_df.columns))

display_cols = [
    col for col in [
        "yyyymm",
        "cell_i",
        "pixel_id",
        "timeline_severity",
        "drought_class",
        "confidence",
        "probabilities",
    ]
    if col in prediction_debug_df.columns
]

print("\nPrediction preview:")
print(prediction_debug_df[display_cols].head(21))

input_cols = [col for col in prediction_debug_df.columns if col.startswith("input_")]
if input_cols:
    print("\nFusion input preview:")
    print(prediction_debug_df[["yyyymm", "pixel_id"] + input_cols].head(21))

Prediction debug rows: 21
Columns:
['yyyymm', 'cell_i', 'pixel_id', 'lon', 'lat', 'timeline_severity', 'severity_label', 'model', 'drought_class_spi3_pred', 'probs', 'confidence', 'gate_weights', 'gate_ranked', 'input_atm_p0', 'input_atm_p1', 'input_atm_p2', 'input_atm_p3', 'input_soil_p0', 'input_soil_p1', 'input_soil_p2', 'input_soil_p3', 'input_veg_p0', 'input_veg_p1', 'input_veg_p2', 'input_veg_p3', 'input_hyd_p0', 'input_hyd_p1', 'input_hyd_p2', 'input_hyd_p3']

Prediction preview:
    yyyymm  cell_i  pixel_id  timeline_severity    confidence
0   202310       0      9405                  5    [0.996042]
1   202310       1      9580                  5  [0.99594456]
2   202310       2      9581                  5   [0.9960337]
3   202311       0      9405                  5    [0.996042]
4   202311       1      9580                  5  [0.99594456]
5   202311       2      9581                  5   [0.9960337]
6   202312       0      9405                  5    [0.996042]
7   202312  

In [14]:
# Cell 13: Inspect raw master inputs for one debug pixel across drought months

DEBUG_PIXEL_ID = 9405
DEBUG_MONTHS = [202310, 202311, 202312, 202401, 202402, 202403, 202404]

pixel_ts_debug_df = (
    master_df[
        (master_df["pixel_id"] == DEBUG_PIXEL_ID)
        & (master_df["yyyymm"].isin(DEBUG_MONTHS))
    ]
    .sort_values("yyyymm")
    .copy()
)

print("Pixel:", DEBUG_PIXEL_ID)
print("Rows found:", len(pixel_ts_debug_df))
print("Columns:")
print(list(pixel_ts_debug_df.columns))
print("\nRaw monthly values:")
print(pixel_ts_debug_df)

Pixel: 9405
Rows found: 0
Columns:
['pixel_id', 'row', 'col', 'lon', 'lat', 'scenario', 'yyyymm', 't2m', 'd2m', 'pet', 'sm', 'ndvi', 'tws']

Raw monthly values:
Empty DataFrame
Columns: [pixel_id, row, col, lon, lat, scenario, yyyymm, t2m, d2m, pet, sm, ndvi, tws]
Index: []


In [15]:
# Cell 13: Diagnose polygon pixel_id versus master_df pixel_id mismatch

DEBUG_PIXEL_ID = 9405
DEBUG_LON = 31.059251
DEBUG_LAT = -17.988764
DEBUG_MONTHS = [202310, 202311, 202312, 202401, 202402, 202403, 202404]

print("Master pixel_id dtype:", master_df["pixel_id"].dtype)
print("Master yyyymm dtype:", master_df["yyyymm"].dtype)
print("Master scenario values:", sorted(master_df["scenario"].dropna().unique().tolist())[:20])

pixel_exact_df = master_df[master_df["pixel_id"] == DEBUG_PIXEL_ID]

print("\nExact pixel_id rows:", len(pixel_exact_df))

nearby_df = (
    master_df[
        (master_df["lon"].between(DEBUG_LON - 0.05, DEBUG_LON + 0.05))
        & (master_df["lat"].between(DEBUG_LAT - 0.05, DEBUG_LAT + 0.05))
        & (master_df["yyyymm"].isin(DEBUG_MONTHS))
    ]
    .sort_values(["yyyymm", "pixel_id"])
    .copy()
)

print("Nearby rows for drought months:", len(nearby_df))
print("Nearby unique pixel_ids:", nearby_df["pixel_id"].drop_duplicates().head(20).tolist())

print("\nNearby sample:")
print(nearby_df.head(20))

Master pixel_id dtype: int64
Master yyyymm dtype: str
Master scenario values: ['historical', 'ssp245', 'ssp370', 'ssp585']

Exact pixel_id rows: 1455
Nearby rows for drought months: 0
Nearby unique pixel_ids: []

Nearby sample:
Empty DataFrame
Columns: [pixel_id, row, col, lon, lat, scenario, yyyymm, t2m, d2m, pet, sm, ndvi, tws]
Index: []


In [16]:
# Cell 14: Inspect raw pixel time series with correct yyyymm string filtering

DEBUG_PIXEL_ID = 9405
DEBUG_MONTHS = ["202310", "202311", "202312", "202401", "202402", "202403", "202404"]

pixel_ts_debug_df = (
    master_df[
        (master_df["pixel_id"] == DEBUG_PIXEL_ID)
        & (master_df["scenario"] == "historical")
        & (master_df["yyyymm"].isin(DEBUG_MONTHS))
    ]
    .sort_values("yyyymm")
    .copy()
)

print("Pixel:", DEBUG_PIXEL_ID)
print("Rows found:", len(pixel_ts_debug_df))
print("Scenario values:", pixel_ts_debug_df["scenario"].unique().tolist())
print("Months found:", pixel_ts_debug_df["yyyymm"].tolist())

print("\nRaw monthly values:")
print(pixel_ts_debug_df)

Pixel: 9405
Rows found: 7
Scenario values: ['historical']
Months found: ['202310', '202311', '202312', '202401', '202402', '202403', '202404']

Raw monthly values:
          pixel_id  row  col        lon        lat    scenario  yyyymm  \
14066280      9405   53  130  31.059251 -17.988764  historical  202310   
14093055      9405   53  130  31.059251 -17.988764  historical  202311   
14119830      9405   53  130  31.059251 -17.988764  historical  202312   
14146605      9405   53  130  31.059251 -17.988764  historical  202401   
14173380      9405   53  130  31.059251 -17.988764  historical  202402   
14200155      9405   53  130  31.059251 -17.988764  historical  202403   
14226930      9405   53  130  31.059251 -17.988764  historical  202404   

                 t2m         d2m       pet        sm      ndvi        tws  
14066280  295.542114  283.900726 -0.000412  0.160912  0.319106 -11.739205  
14093055  295.890289  284.569244 -0.000416  0.128862  0.379721 -11.659789  
14119830  295.2

In [17]:
# Cell 15: Inspect prepared subsystem inputs for one pixel across drought months

DEBUG_PIXEL_ID = 9405
DEBUG_MONTHS = [202310, 202311, 202312, 202401, 202402, 202403, 202404]

prepared_debug_rows = []

for yyyymm in DEBUG_MONTHS:
    ts = (
        master_df[
            (master_df["pixel_id"] == DEBUG_PIXEL_ID)
            & (master_df["scenario"] == "historical")
        ]
        .sort_values("yyyymm")
        .copy()
    )

    print(f"Preparing subsystem inputs | pixel {DEBUG_PIXEL_ID} | {yyyymm}")

    prepared = service.feature_service.prepare_subsystem_inputs(
        ts,
        run_yyyymm=yyyymm,
    )

    for subsystem_name, subsystem_df in prepared.items():
        row = {
            "yyyymm": yyyymm,
            "subsystem": subsystem_name,
            "rows": len(subsystem_df),
            "columns": list(subsystem_df.columns),
        }

        if len(subsystem_df) > 0:
            for col, value in subsystem_df.iloc[0].to_dict().items():
                row[col] = value

        prepared_debug_rows.append(row)

prepared_debug_df = pd.DataFrame(prepared_debug_rows)

print("\nPrepared subsystem debug:")
print(prepared_debug_df)

Preparing subsystem inputs | pixel 9405 | 202310
Preparing subsystem inputs | pixel 9405 | 202311
Preparing subsystem inputs | pixel 9405 | 202312
Preparing subsystem inputs | pixel 9405 | 202401
Preparing subsystem inputs | pixel 9405 | 202402
Preparing subsystem inputs | pixel 9405 | 202403
Preparing subsystem inputs | pixel 9405 | 202404

Prepared subsystem debug:
    yyyymm    subsystem  rows  \
0   198001  atmospheric   555   
1   198001         soil   555   
2   198001   vegetation   555   
3   198001    hydrology   555   
4   198001  atmospheric   555   
5   198001         soil   555   
6   198001   vegetation   555   
7   198001    hydrology   555   
8   198001  atmospheric   555   
9   198001         soil   555   
10  198001   vegetation   555   
11  198001    hydrology   555   
12  198001  atmospheric   555   
13  198001         soil   555   
14  198001   vegetation   555   
15  198001    hydrology   555   
16  198001  atmospheric   555   
17  198001         soil   555   
18 

In [18]:
# Cell 16: Inspect prepared subsystem target-month rows only

DEBUG_PIXEL_ID = 9405
DEBUG_MONTHS = [202310, 202311, 202312, 202401, 202402, 202403, 202404]

target_prepared_rows = []

ts = (
    master_df[
        (master_df["pixel_id"] == DEBUG_PIXEL_ID)
        & (master_df["scenario"] == "historical")
    ]
    .sort_values("yyyymm")
    .copy()
)

for run_yyyymm in DEBUG_MONTHS:
    print(f"Inspecting prepared target rows | pixel {DEBUG_PIXEL_ID} | {run_yyyymm}")

    prepared = service.feature_service.prepare_subsystem_inputs(
        ts,
        run_yyyymm=run_yyyymm,
    )

    for subsystem_name, subsystem_df in prepared.items():
        subsystem_df = subsystem_df.copy()
        subsystem_df["yyyymm_str"] = subsystem_df["yyyymm"].astype(str)

        target_df = subsystem_df[subsystem_df["yyyymm_str"] == str(run_yyyymm)]

        row = {
            "run_yyyymm": run_yyyymm,
            "subsystem": subsystem_name,
            "prepared_rows": len(subsystem_df),
            "target_rows": len(target_df),
            "first_yyyymm": subsystem_df["yyyymm"].iloc[0] if len(subsystem_df) else None,
            "last_yyyymm": subsystem_df["yyyymm"].iloc[-1] if len(subsystem_df) else None,
        }

        if not target_df.empty:
            for col, value in target_df.iloc[0].to_dict().items():
                if col != "yyyymm_str":
                    row[col] = value

        target_prepared_rows.append(row)

target_prepared_debug_df = pd.DataFrame(target_prepared_rows)

print("\nPrepared target-month debug:")
print(target_prepared_debug_df)

Inspecting prepared target rows | pixel 9405 | 202310
Inspecting prepared target rows | pixel 9405 | 202311
Inspecting prepared target rows | pixel 9405 | 202312
Inspecting prepared target rows | pixel 9405 | 202401
Inspecting prepared target rows | pixel 9405 | 202402
Inspecting prepared target rows | pixel 9405 | 202403
Inspecting prepared target rows | pixel 9405 | 202404

Prepared target-month debug:
    run_yyyymm    subsystem  prepared_rows  target_rows first_yyyymm  \
0       202310  atmospheric            555            1       198001   
1       202310         soil            555            1       198001   
2       202310   vegetation            555            1       198001   
3       202310    hydrology            555            1       198001   
4       202311  atmospheric            555            1       198001   
5       202311         soil            555            1       198001   
6       202311   vegetation            555            1       198001   
7       202311  

In [19]:
# Cell 17: Inspect subsystem outputs before fusion for one pixel across drought months

DEBUG_PIXEL_ID = 9405
DEBUG_MONTHS = [202310, 202311, 202312, 202401, 202402, 202403, 202404]

subsystem_output_rows = []

ts = (
    master_df[
        (master_df["pixel_id"] == DEBUG_PIXEL_ID)
        & (master_df["scenario"] == "historical")
    ]
    .sort_values("yyyymm")
    .copy()
)

for run_yyyymm in DEBUG_MONTHS:
    print(f"Running subsystem outputs | pixel {DEBUG_PIXEL_ID} | {run_yyyymm}")

    prepared = service.feature_service.prepare_subsystem_inputs(
        ts,
        run_yyyymm=run_yyyymm,
    )

    subsystem_outputs = service.subsystem_service.run_subsystems(prepared)

    for subsystem_name, output_df in subsystem_outputs.items():
        row = {
            "run_yyyymm": run_yyyymm,
            "subsystem": subsystem_name,
            "rows": len(output_df),
            "columns": list(output_df.columns),
        }

        if len(output_df) > 0:
            for col, value in output_df.iloc[0].to_dict().items():
                row[col] = value

        subsystem_output_rows.append(row)

    gc.collect()

subsystem_output_debug_df = pd.DataFrame(subsystem_output_rows)

print("\nSubsystem output debug:")
print(subsystem_output_debug_df)

Running subsystem outputs | pixel 9405 | 202310
Running subsystem outputs | pixel 9405 | 202311
Running subsystem outputs | pixel 9405 | 202312
Running subsystem outputs | pixel 9405 | 202401
Running subsystem outputs | pixel 9405 | 202402
Running subsystem outputs | pixel 9405 | 202403
Running subsystem outputs | pixel 9405 | 202404

Subsystem output debug:
    run_yyyymm    subsystem  rows           columns        p0            p1  \
0       202310  atmospheric   555  [p0, p1, p2, p3]  0.996043  3.633831e-03   
1       202310         soil   555  [p0, p1, p2, p3]  0.000379  4.696944e-05   
2       202310   vegetation   555  [p0, p1, p2, p3]  0.164095  5.618498e-01   
3       202310    hydrology     1  [p0, p1, p2, p3]  1.000000  3.793571e-10   
4       202311  atmospheric   555  [p0, p1, p2, p3]  0.996043  3.633831e-03   
5       202311         soil   555  [p0, p1, p2, p3]  0.000379  4.696944e-05   
6       202311   vegetation   555  [p0, p1, p2, p3]  0.164095  5.618498e-01   
7      

In [20]:
# Cell 18: Inspect subsystem outputs at the target month position

DEBUG_PIXEL_ID = 9405
DEBUG_MONTHS = [202310, 202311, 202312, 202401, 202402, 202403, 202404]

target_subsystem_rows = []

ts = (
    master_df[
        (master_df["pixel_id"] == DEBUG_PIXEL_ID)
        & (master_df["scenario"] == "historical")
    ]
    .sort_values("yyyymm")
    .reset_index(drop=True)
    .copy()
)

for run_yyyymm in DEBUG_MONTHS:
    print(f"Checking target-position subsystem outputs | pixel {DEBUG_PIXEL_ID} | {run_yyyymm}")

    prepared = service.feature_service.prepare_subsystem_inputs(
        ts,
        run_yyyymm=run_yyyymm,
    )

    subsystem_outputs = service.subsystem_service.run_subsystems(prepared)

    for subsystem_name, prepared_df in prepared.items():
        prepared_df = prepared_df.reset_index(drop=True).copy()
        prepared_df["yyyymm_str"] = prepared_df["yyyymm"].astype(str)

        target_positions = prepared_df.index[
            prepared_df["yyyymm_str"] == str(run_yyyymm)
        ].tolist()

        row = {
            "run_yyyymm": run_yyyymm,
            "subsystem": subsystem_name,
            "target_position": target_positions[0] if target_positions else None,
            "prepared_rows": len(prepared_df),
            "output_rows": len(subsystem_outputs[subsystem_name]),
        }

        if target_positions:
            target_pos = target_positions[0]
            output_df = subsystem_outputs[subsystem_name].reset_index(drop=True)

            if len(output_df) == 1:
                output_row = output_df.iloc[0]
                row["selection_rule"] = "single_output_row"
            elif target_pos < len(output_df):
                output_row = output_df.iloc[target_pos]
                row["selection_rule"] = "target_position_row"
            else:
                output_row = None
                row["selection_rule"] = "target_position_missing"

            if output_row is not None:
                for col, value in output_row.to_dict().items():
                    row[col] = value

        target_subsystem_rows.append(row)

target_subsystem_debug_df = pd.DataFrame(target_subsystem_rows)

print("\nTarget-position subsystem output debug:")
print(target_subsystem_debug_df)

Checking target-position subsystem outputs | pixel 9405 | 202310
Checking target-position subsystem outputs | pixel 9405 | 202311
Checking target-position subsystem outputs | pixel 9405 | 202312
Checking target-position subsystem outputs | pixel 9405 | 202401
Checking target-position subsystem outputs | pixel 9405 | 202402
Checking target-position subsystem outputs | pixel 9405 | 202403
Checking target-position subsystem outputs | pixel 9405 | 202404

Target-position subsystem output debug:
    run_yyyymm    subsystem  target_position  prepared_rows  output_rows  \
0       202310  atmospheric              525            555          555   
1       202310         soil              525            555          555   
2       202310   vegetation              525            555          555   
3       202310    hydrology              525            555            1   
4       202311  atmospheric              526            555          555   
5       202311         soil              526    